In [1]:
import numpy as np
from scipy.stats import norm
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

<div style="background-color:#dbeafe; padding:15px; border-radius:8px; border-left:5px solid #2563eb;">

### **Q1. Two-Proportion Z-Test**

**Control:** 180 conversions / 2,500 users &nbsp;|&nbsp; **Treatment:** 210 conversions / 2,500 users

Perform a full two-proportion z-test. Compute the pooled proportion, standard error, z-statistic, and p-value. State your conclusion at **α = 0.05**.

</div>

<div style="background-color:#f8fafc; padding:13px; border-radius:6px; border-left:4px solid #2563eb; margin-bottom:8px;">

**Thought Process**

**Hypotheses:**  
H0: p1 = p2 (no difference in conversion rates)  
H1: p2 > p1 (treatment converts better) -- one-tailed test

| Step | Formula |
|------|---------|
| Pooled proportion | $\hat{p} = \dfrac{x_1 + x_2}{n_1 + n_2}$ |
| Standard error | $SE = \sqrt{\hat{p}(1-\hat{p})\left(\dfrac{1}{n_1}+\dfrac{1}{n_2}\right)}$ |
| Z-statistic | $z = \dfrac{\hat{p}_2 - \hat{p}_1}{SE}$ |
| p-value (one-tailed) | $p = P(Z > z)$ via `norm.sf(z)` |

Under H0 both groups share a common proportion. We pool both samples to estimate it, then measure how many standard errors the observed difference is away from zero.

</div>

In [2]:
# Given data
x1 = 180   # conversions in control
n1 = 2500  # users in control
x2 = 210   # conversions in treatment
n2 = 2500  # users in treatment
alpha = 0.05

In [3]:
# Individual conversion rates
p1 = x1/n1
p2 = x2/n2

In [4]:
# Pooled proportion: best estimate of p under H0
p_pool = (x1+x2)/(n1+n2)
p_pool

0.078

In [5]:
# Standard error of the difference in proportions
se = np.sqrt(p_pool * (1-p_pool) * (1/n1 + 1/n2))
se

np.float64(0.007585037903662711)

In [6]:
# Z-statistic: how many SEs is the observed difference from 0
z_test = (p2-p1)/se
z_test

np.float64(1.5820619688934416)

In [7]:
# Critical value for one-tailed test at alpha = 0.05
z_crit = norm.ppf(1-alpha)
z_crit

np.float64(1.6448536269514722)

In [8]:
# p-value: P(Z > z_test) under H0  |  sf = survival function = 1 - CDF
pval = norm.sf(z_test)
pval

np.float64(0.05681771225610912)

<div style="background-color:#f0fdf4; padding:12px; border-radius:6px; border-left:4px solid #16a34a;">

**Conclusion**

z_test (1.582) &lt; z_crit (1.645) &nbsp;&nbsp;|&nbsp;&nbsp; p-value (0.057) &gt; alpha (0.05)

**Fail to reject H0.** The 30-conversion lift is not statistically significant at the 5% level -- the treatment shows a trend but the evidence is insufficient to conclude a real difference.

</div>

<div style="background-color:#dbeafe; padding:15px; border-radius:8px; border-left:5px solid #2563eb;">

### **Q2. Sample Size Calculation**

Your e-commerce site has a baseline conversion rate of 8%. You want to detect a lift of 1.5 percentage points with 80% power and α = 0.05 (two-tailed). Calculate the required sample size per group. How does your answer change if you want 90% power instead?

</div>

<div style="background-color:#f8fafc; padding:13px; border-radius:6px; border-left:4px solid #2563eb; margin-bottom:8px;">

**Thought Process**

**Hypotheses:**  
H0: p1 = p2 (no difference)  
H1: p1 != p2 (two-tailed; we want to detect any 1.5 pp shift)

| Component | Formula |
|-----------|--------|
| Average proportion | $\bar{p} = (p_1 + p_2)/2$ |
| Numerator | $\left(z_{\alpha/2}\sqrt{2\bar{p}(1-\bar{p})} + z_{\beta}\sqrt{p_1(1-p_1)+p_2(1-p_2)}\right)^2$ |
| Sample size per group | $n = \text{numerator} / (p_2 - p_1)^2$ |

Higher power (80% to 90%) reduces beta, increasing z_beta and therefore requiring more users per group.

</div>

In [9]:
p1 = 0.08
p2 = 0.095
p = (p1+p2)/2
alpha = 0.05

case 1

In [10]:
z_crit = norm.ppf(1-alpha/2)  # two-tailed critical value at alpha=0.05
z_crit

np.float64(1.959963984540054)

In [11]:
beta = 0.20               # beta = 1-power 
z_beta = norm.ppf(1-beta) # z-score for 80% power
z_beta

np.float64(0.8416212335729143)

In [12]:
num = ((z_crit * np.sqrt(2*p*(1-p))) + (z_beta * ( np.sqrt( p1 * (1-p1) + p2 * (1 - p2)))))**2  # numerator of sample size formula
deno = (p2-p1)**2  # denominator: squared effect size

In [13]:
n = num/deno  # required sample size per group at 80% power
n

np.float64(5569.345284898933)

case 2

In [14]:
beta = 0.10
z_beta = norm.ppf(1-beta)  # z-score for 90% power; larger than 80% case
z_beta

np.float64(1.2815515655446004)

In [15]:
num = ((z_crit * np.sqrt(2*p*(1-p))) + (z_beta * ( np.sqrt( p1 * (1-p1) + p2 * (1 - p2)))))**2  # numerator recalculated with 90% power z-score
deno = (p2-p1)**2  # denominator unchanged: same effect size

In [16]:
n2 = num/deno  # required sample size per group at 90% power -- ~34% more users than 80% case
n2

np.float64(7455.2743390574915)

<div style="background-color:#f0fdf4; padding:12px; border-radius:6px; border-left:4px solid #16a34a;">

**Conclusion**

80% power: **n = 5,570 per group** &nbsp;&nbsp;|&nbsp;&nbsp; 90% power: **n = 7,456 per group**

Increasing power from 80% to 90% requires approximately 34% more users per group, reflecting the higher z_beta needed to reduce the false-negative rate.

</div>